# How one country gets its risk rating

This notebook rates a single country from scratch and shows its work: what
evidence went in, how old that evidence is, what the news added, what the model
decided, and why. It runs the same code paths the daily ETL runs — nothing here
is a re-implementation.

**Scores come from three ledgers plus one reading that is not risk at all.**

| Ledger | The question it asks | A high number means |
|---|---|---|
| **Friction** | What does the state extract, and how much of it converts into capability? | a worse wedge |
| **Order-uncertainty** | Are the load-bearing rules — contracts, currency, statistics, succession — legible? | less underwritable |
| **Information** | Can the country's own instruments be trusted to measure it? | weaker instruments |
| **Edge vitality** | Is the system still learning — firms forming, failing, inventing? | *more vitality — and this one is never risk* |

Two rules run through every step:

- **Nothing edits the model's score.** There were floors, a cap and a sanctions
  gate. They are gone. A sanctioned country keeps the score its evidence earned
  and gains a `RESTRICTED` badge beside it. Step 4 proves the score was untouched.
- **Absent means absent.** Every value carries `as_of` and `staleness_days`;
  anything missing is simply not in the payload — never a zero, never a padded null.

**To run:** set `ISO2` in the *Pick a country* cell, then Run All. It makes real
network calls and **spends OpenAI credits** (one cheap digest call per article,
then one scoring call). It writes **no snapshot** — the only thing it touches in
Postgres is the digest cache, which nothing on the dashboard reads.

Kernel, once, into the project venv (deliberately not in `requirements.txt`,
which is runtime-only):

```
.venv\Scripts\python.exe -m pip install ipykernel
```

In [1]:
import json
import logging
import os
import pathlib
import sys
from statistics import median

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display

# Repo root = the folder holding backend/main.py, so this works from any cwd.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

# force=True is not optional: Jupyter installs its own root handler, so a plain
# basicConfig() is silently a no-op and no pipeline logs ever appear.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s",
                    stream=sys.stdout, force=True)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

from backend.util import constants, lint, metrics, policy, provenance
from backend.llm import client as ai_client
from backend.llm import constants as ai_constants
from backend.llm import digest_engine, langchain_llm
from backend.llm import payload as llm_payload
from backend.data_fetching import (
    bis_bulk_fetch, country_data_fetch, curated_loader, imf_macro_fetch, wb_series_fetch,
)
from backend.data_upsert import data_push
from backend.news_fetching import article_enrichment, article_ranking

# Mirrored from backend/util/pipeline.py.
SINCE_YEAR, LOOKBACK_YEARS, DELTA_HORIZONS, MAX_ARTICLES = 2015, 10, (1, 5), 20

for key in ("OPENAI_API_KEY", "CRAWLBASE_TOKEN", "DATABASE_URL"):
    print(f"  {key:<20} {'set' if os.getenv(key) else 'MISSING'}")
print(f"\nscoring model {ai_client.MODEL_NAME}   digest model {ai_client.DIGEST_MODEL_NAME}")
print(f"prompt {ai_constants.PROMPT_VERSION}   policy {policy.POLICY_VERSION}   "
      f"seed {ai_client.SEED}")
print(f"{len(constants.INDICATOR_REGISTRY)} indicators in the registry across 4 ledgers")

  OPENAI_API_KEY       set
  CRAWLBASE_TOKEN      set
  DATABASE_URL         set

scoring model gpt-4o-2024-08-06   digest model gpt-4o-mini-2024-07-18
prompt v4.0-masked-production   policy p2.0-observe-only   seed 42
38 indicators in the registry across 4 ledgers


In [2]:
# --- chart kit -----------------------------------------------------------
# No plotting library: matplotlib is not in the venv and the kernel is kept out
# of requirements.txt, so a chart here must not add a dependency every reader
# has to install. Inline SVG through IPython.display renders in JupyterLab,
# VS Code and GitHub's notebook viewer alike.
#
# Categorical slots come from a validated palette and are assigned by LEDGER
# IDENTITY, never cycled: a ledger keeps its hue in every chart below. Charts
# that are not about a ledger use one neutral fill rather than borrowing a hue
# that means something else. (Validated light and dark: lightness band, chroma
# floor, CVD separation and normal-vision floor all pass. On the light surface
# two slots sit under 3:1, which is why every bar carries a visible value label
# — the labels are relief, not decoration.)

LEDGER_HUE = {"friction": 1, "uncertainty": 2, "information": 3, "edge": 4}
LEDGER_LABEL = {"friction": "friction", "uncertainty": "order-uncertainty",
                "information": "information", "edge": "edge vitality"}

# Bands from the prompt's own calibration (ai/constants.py), on the 0-100 grid.
BANDS = [(0, 20, "Low"), (20, 40, "Low-Moderate"), (40, 75, "Moderate"),
         (75, 90, "High"), (90, 100, "Extreme")]

_CSS = """<style>
.viz{color-scheme:light;--s:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--rule:#e5e4e1;
--c1:#2a78d6;--c2:#eb6834;--c3:#1baf7a;--c4:#eda100;--c0:#6f6e6a;--crit:#d03b3b;
--t1:#e8e7e4;--t2:#d5d4d0;--t3:#bab9b4;--t4:#9a9994;--t5:#7a7975;
font:12px/1.45 ui-sans-serif,-apple-system,"Segoe UI",sans-serif;background:var(--s);
border:1px solid var(--rule);border-radius:6px;padding:14px 16px;margin:4px 0;max-width:880px}
@media (prefers-color-scheme:dark){.viz{color-scheme:dark;--s:#1a1a19;--ink:#fff;
--ink2:#c3c2b7;--rule:#3a3a37;--c1:#3987e5;--c2:#d95926;--c3:#199e70;--c4:#c98500;
--c0:#8f8e88;--t1:#2f2f2c;--t2:#3d3d39;--t3:#4f4f4a;--t4:#64645e;--t5:#7d7d76}}
.viz h4{margin:0;font-size:13px;font-weight:600;color:var(--ink)}
.viz .sub{margin:3px 0 12px;font-size:11px;color:var(--ink2)}
.viz .lg{display:flex;flex-wrap:wrap;gap:14px;margin-top:11px;font-size:11px;color:var(--ink2)}
.viz .lg i{width:9px;height:9px;border-radius:2px;display:inline-block;margin-right:5px;
vertical-align:-1px}
.viz .tiles{display:flex;flex-wrap:wrap;gap:10px;align-items:stretch}
.viz .tile{flex:1 1 130px;border:1px solid var(--rule);border-radius:5px;padding:9px 11px}
.viz .tile .k{font-size:10px;letter-spacing:.06em;text-transform:uppercase;color:var(--ink2)}
.viz .tile .v{font-size:19px;font-weight:600;color:var(--ink);margin-top:3px;
font-variant-numeric:tabular-nums}
.viz .tile .u{font-size:10px;color:var(--ink2);margin-top:1px}
.viz .op{flex:0 0 auto;align-self:center;font-size:16px;color:var(--ink2);padding:0 1px}
.viz .hero{font-size:46px;font-weight:600;color:var(--ink);line-height:1;
font-variant-numeric:tabular-nums}
.viz .band{font-size:13px;color:var(--ink);font-weight:600;margin-top:5px}
.viz .badge{display:inline-block;border:1px solid var(--crit);color:var(--crit);
border-radius:3px;padding:1px 6px;font-size:10px;font-weight:600;letter-spacing:.06em;
margin-left:8px;vertical-align:6px}
.viz .why{margin-top:12px}
.viz .why h5{margin:11px 0 3px;font-size:11px;font-weight:600;color:var(--ink)}
.viz .why h5 i{width:9px;height:9px;border-radius:2px;display:inline-block;margin-right:6px}
.viz .why ul{margin:0;padding-left:22px;color:var(--ink2)}
.viz .why li{margin:2px 0}
.viz text{font:11px ui-sans-serif,-apple-system,"Segoe UI",sans-serif}
</style>"""

_BAR_H, _BAND_H = 14, 26          # <=24px thick marks, and the leftover is air
_PLOT_W, _VAL_W, _CH, _LBL_MAX = 460, 78, 5.7, 268


def _esc(s):
    return (str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))


def _fit(label):
    """Truncate a label to the column width, keeping the full text for hover."""
    room = int(_LBL_MAX / _CH)
    return label if len(label) <= room else label[:room - 1].rstrip() + "…"


def _wrap(title, sub, body, footer=""):
    return HTML(_CSS + f'<div class="viz"><h4>{_esc(title)}</h4>'
                + (f'<p class="sub">{sub}</p>' if sub else "") + body
                + (f'<p class="sub" style="margin:9px 0 0">{footer}</p>' if footer else "")
                + "</div>")


def hbar(rows, title, sub="", *, fmt="{:.2f}", max_value=None, footer=""):
    """Horizontal bars: rows of (label, value, ledger_key_or_None, hover_note).

    One baseline, thin marks, a 4px rounded data-end squared off at the
    baseline, and a direct value label on every bar. `None` as the key uses the
    neutral fill - for charts that are not about a ledger.
    """
    rows = [r for r in rows if r[1] is not None]
    if not rows:
        return _wrap(title, "no values available", "")
    top = max_value or max(abs(v) for _, v, _, _ in rows) or 1.0
    lbl_w = min(_LBL_MAX, max(70, int(max(len(_fit(r[0])) for r in rows) * _CH) + 6))
    x0, h = lbl_w + 10, _BAND_H * len(rows) + 6
    out = [f'<svg width="{x0 + _PLOT_W + _VAL_W}" height="{h}" role="img">',
           f'<line x1="{x0}" y1="2" x2="{x0}" y2="{h - 4}" stroke="var(--rule)"/>']
    for i, (label, value, key, note) in enumerate(rows):
        y = i * _BAND_H + 3
        w = max(2.0, abs(value) / top * _PLOT_W)
        c = f"var(--c{LEDGER_HUE.get(key, 0)})"
        out += [f'<g><title>{_esc(label)}: {fmt.format(value)}{_esc(note)}</title>',
                f'<text x="{lbl_w}" y="{y + 11}" text-anchor="end" fill="var(--ink2)">'
                f'{_esc(_fit(label))}</text>',
                f'<rect x="{x0}" y="{y}" width="{w:.1f}" height="{_BAR_H}" rx="4" fill="{c}"/>',
                # rx rounds both ends; this squares the one anchored to the baseline.
                f'<rect x="{x0}" y="{y}" width="{min(4.0, w):.1f}" height="{_BAR_H}" fill="{c}"/>',
                # Value in text ink, never the series colour.
                f'<text x="{x0 + w + 7:.1f}" y="{y + 11}" fill="var(--ink)" '
                f'style="font-variant-numeric:tabular-nums">{fmt.format(value)}</text></g>']
    out.append("</svg>")
    seen = [k for k in dict.fromkeys(r[2] for r in rows) if k]
    legend = ""
    if len(seen) > 1:      # one series needs no legend box - the title names it
        legend = '<div class="lg">' + "".join(
            f'<span><i style="background:var(--c{LEDGER_HUE[k]})"></i>{LEDGER_LABEL[k]}</span>'
            for k in seen) + "</div>"
    return _wrap(title, sub, "".join(out) + legend, footer)


def tiles(items, title, sub=""):
    """Stat tiles for the numbers that are a sentence, not a chart."""
    body = []
    for it in items:
        if it in ("x", "="):
            body.append(f'<div class="op">{"&times;" if it == "x" else "="}</div>')
        else:
            k, v, u = it
            body.append(f'<div class="tile"><div class="k">{_esc(k)}</div>'
                        f'<div class="v">{_esc(v)}</div><div class="u">{_esc(u)}</div></div>')
    return _wrap(title, sub, '<div class="tiles">' + "".join(body) + "</div>")


def scorecard(name, as_of, score_12m, score_3m, *, non_investable=False, summary=""):
    """The headline: one number is a hero, not a chart.

    Both horizons sit on ONE 0-100 axis over the prompt's own bands, so the
    3-month and 12-month readings are compared against each other rather than
    against two scales. The band track is neutral by design - a hue there would
    impersonate a ledger, and the band is named in text anyway.
    """
    def pct(v):
        return None if v is None else round(v * 100)

    v12, v3 = pct(score_12m), pct(score_3m)
    band = next((n for lo, hi, n in BANDS if v12 is not None and lo <= v12 < hi), "n/a")
    W, X0, Y = 560, 4, 43
    seg = [f'<rect x="{X0 + lo / 100 * W:.1f}" y="{Y}" width="{(hi - lo) / 100 * W - 2:.1f}" '
           f'height="9" rx="2" fill="var(--t{i + 1})"><title>{lo}-{hi} {n}</title></rect>'
           for i, (lo, hi, n) in enumerate(BANDS)]
    ticks = [f'<text x="{X0 + lo / 100 * W:.1f}" y="{Y + 24}" fill="var(--ink2)" '
             f'font-size="10">{n}</text>' for lo, _, n in BANDS]
    # When the horizons land close together their labels would overlap, so the
    # 3-month one steps up a line. The markers themselves stay on the axis.
    close = v12 is not None and v3 is not None and abs(v12 - v3) < 11
    marks = []
    for v, label, solid, lift in ((v12, "12m", True, 0), (v3, "3m", False, 13 if close else 0)):
        if v is None:
            continue
        x = X0 + v / 100 * W
        marks += [f'<line x1="{x:.1f}" y1="{Y - 7}" x2="{x:.1f}" y2="{Y + 13}" '
                  f'stroke="var(--ink)" stroke-width="2"/>',
                  # 2px surface ring so the two markers stay separable when close.
                  f'<circle cx="{x:.1f}" cy="{Y - 9}" r="5" fill="'
                  + ("var(--ink)" if solid else "var(--s)")
                  + '" stroke="var(--ink)" stroke-width="2"/>',
                  # Clamped so a 0 or a 100 keeps its label inside the viewport.
                  f'<text x="{min(max(x, 24), X0 + W + 26):.1f}" y="{Y - 18 - lift}" '
                  f'text-anchor="middle" fill="var(--ink)" font-size="11" '
                  f'style="font-variant-numeric:tabular-nums">{label} {v}</text>']
    badge = '<span class="badge">RESTRICTED</span>' if non_investable else ""
    body = (f'<div class="hero">{v12 if v12 is not None else "--"}'
            f'<span style="font-size:15px;color:var(--ink2)"> / 100</span>{badge}</div>'
            f'<div class="band">{band} risk over 12 months'
            + (f' &middot; {v3} over 3 months' if v3 is not None else "") + "</div>"
            + f'<svg width="{W + 60}" height="{Y + 34}" role="img" style="margin-top:14px">'
            + "".join(seg + ticks + marks) + "</svg>"
            + (f'<p class="sub" style="margin:10px 0 0;color:var(--ink);white-space:pre-wrap">'
               f'{_esc(summary)}</p>' if summary else ""))
    return _wrap(f"{name} - risk rating as of {as_of}",
                 "filled marker = 12-month horizon, hollow = 3-month. Bands are the "
                 "prompt's own calibration anchors.", body)


def notes(groups, title, sub=""):
    """A ledger-coloured chip beside each list: the model's own citations."""
    body = ['<div class="why">']
    for key, heading, lines in groups:
        if not lines:
            continue
        body.append(f'<h5><i style="background:var(--c{LEDGER_HUE[key]})"></i>{_esc(heading)}</h5>'
                    "<ul>" + "".join(f"<li>{_esc(x)}</li>" for x in lines) + "</ul>")
    body.append("</div>")
    return _wrap(title, sub, "".join(body))


print("chart kit ready - inline SVG, no plotting dependency")

chart kit ready - inline SVG, no plotting dependency


## Pick a country

The annual macro panel used to be a Parquet partition under `backend/data`. It
now lives in Postgres like every other observation, so there is **one** macro
store rather than two with different vintage semantics — and a clone with an
empty database builds it by fetching rather than by finding files.

`has_series` says whether anything is stored for this country yet. A country
with nothing triggers a slow backfill in the next step — one call per indicator.


In [3]:
ISO2 = "PT"   # <-- change country here

ENTRY = next(c for c in constants.COUNTRY_ROSTER if c["iso2"] == ISO2)
NAME, ISO3 = ENTRY["name"], ENTRY["iso3"]

# One read, reused below. An empty dict is the honest answer for a fresh clone
# and for a database that is simply down — both mean "nothing stored", and the
# next cell fetches in either case.
try:
    STORED_SERIES = data_push.read_indicator_series(ISO2) or {}
except Exception as exc:
    print(f"indicator_series unavailable ({type(exc).__name__}) - treating as empty")
    STORED_SERIES = {}
HAS_SERIES = bool(STORED_SERIES)

print(f"{NAME}  ({ISO2}/{ISO3})  tier={ENTRY['tier']}  "
      f"macro series {f'stored ({len(STORED_SERIES)} indicators)' if HAS_SERIES else 'MISSING - will be fetched (slow)'}")

roster = pd.DataFrame(constants.COUNTRY_ROSTER)[["name", "iso2", "iso3", "tier"]]
print(f"{len(roster)} countries in the roster")
display(roster.head(10))


Portugal  (PT/PRT)  tier=DM  macro series stored (22 indicators)
48 countries in the roster


,name,iso2,iso3,tier
0,Australia,AU,AUS,DM
1,Austria,AT,AUT,DM
2,Belgium,BE,BEL,DM
3,Canada,CA,CAN,DM
4,Denmark,DK,DNK,DM
5,Finland,FI,FIN,DM
6,France,FR,FRA,DM
7,Germany,DE,DEU,DM
8,"Hong Kong SAR, China",HK,HKG,DM
9,Ireland,IE,IRL,DM


## 1 · The evidence

**Two stores feed the model**, and for each indicator the *freshest period wins*:

| Store | What it holds |
|---|---|
| `indicator_series` | every observation at any frequency — World Bank annuals, BIS, IMF monthlies, and the hand-typed rows in `backend/data/curated.csv` |
| `structural_facts.yaml` | the static half: currency-union membership, reserve-currency status, things that are structure rather than reputation |

This used to be three. A Parquet macro panel held the World Bank annuals and a
`recent_indicator` table held one latest sub-annual print each, and the two had
different vintage semantics — so the same number could be stale in one store and
current in the other. Both are gone: the annuals moved into `indicator_series`
and `recent_indicator` was dropped, because `_resolve` already picks the freshest
observation out of a full history and did not need a second table to do it.

Freshest-period-wins is the point of the whole rebuild. Before it, the model
scored Argentina's inflation on a World Bank annual average up to two years stale
while the front-end, reading the same database, showed the current monthly print.

Everything is passed *into* `build_evidence_payload` rather than read inside it,
so the builder is pure and re-runnable over history — and each read degrades on
its own. No database, or no curated rows, costs the country that evidence and
not its score.


In [4]:
def safe(read, what):
    """Mirror the pipeline's per-store resilience: a failed read is absent, not fatal."""
    try:
        return read()
    except Exception as exc:
        print(f"  {what:<18} unavailable ({type(exc).__name__}) - degrading to absent")
        return None


# --- the annual panel, backfilled only if this country has none --------------
if not HAS_SERIES:
    full_roster = constants.COUNTRY_ROSTER
    constants.COUNTRY_ROSTER = [ENTRY]          # the narrowing live_country_check.py uses
    try:
        country_data_fetch.backfill_missing_panels()
    finally:
        constants.COUNTRY_ROSTER = full_roster

# --- the panel payload -------------------------------------------------------
# The scorer does NOT read this one. It stays because upsert_snapshot reads its
# `indicators` and `_meta.units` to write the tables the front-end's indicator
# pane queries, and provenance reads its `series`. Reshaping it into ledgers
# would have broken the front-end silently.
payload = llm_payload.prepare_llm_payload_pretty(
    country_iso=ISO2, indicators=constants.ALL_INDICATORS,
    since=SINCE_YEAR, lookback=LOOKBACK_YEARS, deltas=DELTA_HORIZONS,
)
AS_OF = data_push.payload_as_of(payload)        # the date the snapshot is keyed on

# --- the sub-annual and curated stores --------------------------------------
FETCH_MISSING_SERIES = True   # False = score on whatever is already stored
FETCH_BIS = True              # ~14 MB of BIS bulk files; the only source of FX volatility

series = safe(lambda: data_push.read_indicator_series(ISO2), "indicator_series") or {}
if not series and FETCH_MISSING_SERIES:
    print(f"no stored series for {ISO2} - fetching (World Bank, IMF"
          f"{', BIS bulk' if FETCH_BIS else ''}, curated.csv)")
    fetched = []
    fetched += wb_series_fetch.fetch_country_series(ISO2, ISO3, as_of=AS_OF) or []
    fetched += imf_macro_fetch.fetch_series_rows(ISO2, ISO3, as_of=AS_OF) or []
    if FETCH_BIS:
        for code in ("BIS.POLICY.RATE", "BIS.FX.USD"):
            fetched += [r for r in bis_bulk_fetch.fetch_dataset_rows(code, as_of=AS_OF)
                        if r["country_iso2"] == ISO2]
    fetched += [r for r in safe(curated_loader.load_curated_series, "curated.csv") or []
                if r["country_iso2"] == ISO2]
    series = {}
    for r in sorted(fetched, key=lambda r: (r["indicator_code"], r["period"])):
        series.setdefault(r["indicator_code"], []).append(r)
    if os.getenv("DATABASE_URL") and fetched:
        safe(lambda: data_push.upsert_indicator_series(fetched), "series upsert")

# `panel` and `recent` were separate arguments until both stores were folded
# into `indicator_series`. `structural` is the block that replaced neither: it is
# the static half of the evidence, and it is what lets a MASKED run keep the
# priors the country's name used to carry — that a currency-union member cannot
# devalue, that a reserve-currency issuer's fiscal arithmetic differs in kind.
evidence = llm_payload.build_evidence_payload(
    ISO2, as_of=AS_OF, series=series,
    fx_regimes=constants.FX_REGIMES, elections=constants.ELECTIONS,
    structural=safe(curated_loader.load_structural_facts, "structural") or {},
)

SECTIONS = (("friction_inputs", "friction"), ("uncertainty_inputs", "uncertainty"),
            ("information_inputs", "information"), ("edge_inputs", "edge"))
present = sum(len(evidence[s]) for s, _ in SECTIONS)
print(f"\nas_of {AS_OF} - {present} indicators present, "
      f"{len(evidence['computed'])} computed metrics, "
      f"~{len(json.dumps(evidence, ensure_ascii=False)) // 4} tokens")

INFO    backend.data_fetching.curated_loader: [structural] 5 of 48 countries have a structural block


INFO    backend.llm.payload: [PT] evidence payload ~1924 tokens (budget 2800)



as_of 2026-08-15 - 23 indicators present, 7 computed metrics, ~1924 tokens


### How old is the evidence?

`staleness_days` counts from the **end of the period a value describes** to
`as_of` — how old the reading is. That is a different fact from the value's
`as_of`, which is when it became known to us. Conflating the two is exactly how
a stale number passes for a current one.

But raw age misleads on its own. A World Bank annual series at 574 days is doing
its job — the 2024 round is the newest that exists. So the chart divides age by
**how often the source republishes**: 1.0× arrived on schedule, 2.0× means a full
cycle came and went with nothing new. Only those are worth discounting, and the
prompt tells the model to.

In [5]:
CADENCE = {"M": 30, "Q": 91, "A": 365}
# Two series declare freq "A" because their period is a year, yet neither
# republishes annually - the HCI arrives in irregular rounds (2017/2018/2020),
# PISA on a fixed 3-year cycle. Keyed by `source` because that is what a payload
# entry carries; the registry code never reaches the payload at all.
CYCLE_DAYS = {"World Bank Human Capital Project": 365 * 3, "OECD PISA": 365 * 3}

entries = []
for section, key in SECTIONS:
    found = [(n, e, key) for n, e in evidence[section].items()
             if isinstance(e, dict) and e.get("staleness_days") is not None]
    entries += sorted(found, key=lambda t: t[1]["staleness_days"])   # freshest first

behind = [(name[:46],
           e["staleness_days"] / CYCLE_DAYS.get(e["source"], CADENCE[e["freq"]]), key,
           f" · {e['staleness_days']}d · {e['period']} ({e['freq']}) · {e['source']}")
          for name, e, key in entries]
late = sum(1 for _, cycles, _, _ in behind if cycles >= 2.0)

display(hbar(
    behind, f"Publication cycles behind - {NAME} as of {AS_OF}",
    "evidence age divided by how often the source republishes. Hover for raw age, "
    "period and source.",
    fmt="{:.1f}×",
    footer=f"{late} of {len(behind)} indicators sit at 2.0× or worse. Anything the "
           f"registry defines but no store has is absent from the payload entirely, "
           f"never a padded null.",
))

display(tiles(
    [(LEDGER_LABEL[key],
      f"{median([e['staleness_days'] for _, e, k in entries if k == key]):.0f}d",
      f"{sum(1 for _, _, k in entries if k == key)} indicators · oldest "
      f"{max(e['staleness_days'] for _, e, k in entries if k == key)}d")
     for key in dict.fromkeys(k for _, _, k in entries)],
    "Median evidence age by ledger",
    "edge sits oldest for every country, not just this one: learning outcomes and "
    "human capital are measured on multi-year assessment cycles, and no source "
    "anywhere reports them faster.",
))

### What the arithmetic already knows

Code does the deterministic sums so the model does not have to do them in its
head. Each is a pure function in `utils/metrics.py`, and any missing input yields
`None` rather than a fabricated zero — a metric absent from this table is a
statement about our evidence, not about the country.

In [6]:
computed = evidence["computed"]
friction = evidence["friction_inputs"]

# The headline measure is a sentence, not a chart: what the state takes, times
# how badly it converts, is the wedge.
take = friction.get("Tax revenue (% GDP)", {}).get("value")
loss, wedge = computed.get("conversion_loss"), computed.get("frictional_extraction")
if wedge is not None:
    display(tiles(
        [("extraction", f"{take:.1f}%", "tax revenue, % GDP"), "x",
         ("conversion loss", f"{loss:.3f}", "0-1, from GE z-score + corruption"), "=",
         ("the wedge", f"{wedge:.2f}%", "of GDP taken and lost")],
        "Frictional extraction",
        "Judge the take by how it converts, not by its size: a large take that funds "
        "functioning courts and registries is not friction; a modest one that funds "
        "nothing is.",
    ))

# The purity claim, made checkable: conversion_loss took two published numbers and
# nothing else, so recomputing it here must land on the same value. No clock, no
# network - which is what lets the whole layer be re-run over history for free.
ge = friction.get("Government effectiveness (z-score)", {}).get("value")
corr = friction.get("Political corruption index (0–1, higher = more corrupt)", {}).get("value")
if loss is not None:
    again = metrics.conversion_loss(ge, corr)
    print(f"conversion_loss({ge}, {corr}) = {again}   stored {loss}   "
          f"{'match' if again == loss else 'MISMATCH'}")

flat = {}
for k, v in computed.items():
    flat.update({f"{k}.{kk}": vv for kk, vv in v.items()} if isinstance(v, dict) else {k: v})
display(pd.Series(flat, name="value").to_frame())

flag = evidence["uncertainty_inputs"]["suppressed_vol_flag"]
print(f"suppressed_vol_flag  {flag['value']!s:<6} regime={flag['regime']!s:<8} "
      f"fx_vol={flag['fx_volatility_24m']!s:<10} reserves_6m={flag['reserves_trend_6m']}")
print("  null is not False - it means one of the three inputs is unavailable.")
print("  When true, the prompt tells the model to read measured calm as evidence")
print("  AGAINST the country: a defended peg draining reserves is fuel load.")

conversion_loss(0.9475, 0.166) = 0.2383   stored 0.2383   match


,value
conversion_loss,0.2383
frictional_extraction,5.3254
doom_loop.burden_5y_delta,-0.1
doom_loop.conversion_quality_5y_delta,-0.0291
doom_loop.burden_up_quality_down,False
cpi_volatility_36m,0.538713
fx_volatility_24m,2.163258
precommitted_share.value,5.4069
precommitted_share.partial,True
dependency_trajectory.current,40.0084


suppressed_vol_flag  None   regime=None     fx_vol=2.163258   reserves_6m=None
  null is not False - it means one of the three inputs is unavailable.
  When true, the prompt tells the model to read measured calm as evidence
  AGAINST the country: a defended peg draining reserves is fuel load.


## 2 · The news

Six Google News queries per country — one for each ledger the prompt scores
(friction, order, security, information, edge) plus a broad catch-all —
de-duplicated by publisher URL and scored by a keyword heuristic. Each theme is
guaranteed a share of the 20-article budget, so an election week cannot crowd
out the tax and press-freedom stories the friction and information ledgers need.
Google's links are
redirect wrappers, so survivors get unwrapped to real publisher URLs, denylisted
sources dropped, and one GET each recovers body text and a thumbnail.

Then the funnel narrows twice more. A cheap model reads each article's **full
text** and returns a strict-JSON factual extraction plus a 0-100 severity; every
digest reaches the scorer, but only the three highest-severity articles are pasted
in full. Digests are cached in Postgres by `(country, as_of, url)` plus a hash of
the text, so re-running this cell on the same day costs nothing.

In [7]:
items = article_enrichment.fetch_relevant_news(NAME, max_articles=MAX_ARTICLES)
fetched_n = len(items)

items = article_enrichment.resolve_and_enrich(items, ISO2)
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"                      # the stable ids the model cites back

items = digest_engine.digest_articles(items, country_display=NAME, iso2=ISO2, as_of=AS_OF)
fulltext_ids = digest_engine.select_fulltext_ids(items)
digested = [it for it in items if it.get("digest")]

display(hbar(
    [("fetched", fetched_n, None, " · 4 queries, de-duped, relevance-scored"),
     ("survived enrichment", len(items), None, " · denylist + publisher URL resolved"),
     ("digested", len(digested), None, f" · one {ai_client.DIGEST_MODEL_NAME} call each"),
     ("read in full by the scorer", len(fulltext_ids), None,
      f" · {', '.join(fulltext_ids)}")],
    f"How {fetched_n} headlines become {len(fulltext_ids)} the scorer reads closely",
    "every digest still reaches the scorer - the last step is about depth, not inclusion.",
    fmt="{:.0f}",
    footer="Nothing here raises: a per-article failure leaves digest=None and that "
           "article degrades to its title and summary in the prompt.",
))

pd.DataFrame([
    {"id": it["id"], "severity": it.get("stage1_severity"),
     "full_text": it["id"] in fulltext_ids, "source": it.get("source"),
     "what_happened": (it.get("digest") or {}).get("what_happened")}
    for it in items
]).sort_values("severity", ascending=False).set_index("id")

INFO    backend.news_fetching.source_filter: Loaded 3 blocked news source(s).


INFO    httpx: HTTP Request: GET https://www.theportugalnews.com/news/2026-08-14/us-llc-income-in-portugal-new-tax-ruling-brings-long-awaited-clarity/1069174 "HTTP/1.1 200 OK"


INFO    httpx: HTTP Request: GET https://portugoal.net/club-news/5911-end-of-the-line-for-boavista-as-court-ruling-forces-club-to-close-down "HTTP/1.1 200 OK"


INFO    httpx: HTTP Request: GET https://www.stocktitan.net/news/BKYI/portuguese-national-security-agency-selects-bio-key-and-visualforma-qjdk00md72yc.html "HTTP/1.1 200 OK"


INFO    httpx: HTTP Request: GET https://www.techtimes.com/articles/324369/20260813/portugal-solar-auctions-never-protected-farmland-agriculture-minister-cant-force-change.htm "HTTP/1.1 200 OK"


...

INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO    backend.llm.digest_engine: [PT] digests: ok=20 cached=0 content-cached=0 failed=0


,severity,full_text,source,what_happened
id,,,,
a19,85.0,True,Outside Magazine,British cyclist Finlay Tarling died after succumbing to injuries sustained i...
a16,75.0,True,NewsCord,19-year-old Welsh cyclist Finlay Tarling died after a crash during the eight...
a2,60.0,True,Portugal Resident,"Portugal's minister for internal administration, Luís Neves, announced new m..."
a10,60.0,False,portugoal.net,Boavista Futebol Clube received a court order to shut down all their activit...
a12,40.0,False,BBC,There have been warnings of queues at airports in Europe this summer because...
a4,40.0,False,The Portugal News,The Portuguese Tax Authority provided clarification on the tax treatment of ...
a1,40.0,False,The Korea Herald,Portugal transitioned from a military dictatorship to a democracy after the ...
a6,40.0,False,Stock Titan,BIO-key International and Visualforma were awarded a contract to enhance dig...
a7,40.0,False,Tech Times,Portugal's Agriculture Minister José Manuel Fernandes stated that solar pane...


### Breadth from the digests, depth from three

The mini model is an **extraction engine**, not an analyst: it is told to use only
the text in front of it and to write `"not stated"` rather than fill a gap from
outside knowledge. It never sees the macro payload, the other articles, or the
scoring rubric, and it never produces a risk score. The one judgement it makes —
`stage1_severity` — is used for exactly one thing: choosing which three articles
the scorer reads in full.

The chart below is the scorer's attention budget. The digests are the compression
that makes breadth affordable.

In [8]:
evidence_json = json.dumps(evidence, ensure_ascii=False)
digests_json = langchain_llm._digests_to_json(items)            # the prompt's own builders,
fulltext = langchain_llm._fulltext_block(items, fulltext_ids)   # not a re-implementation
all_bodies = sum(len(digest_engine.article_input_text(it)) for it in items)

display(hbar(
    [("EVIDENCE_JSON", len(evidence_json), None, " · the three-ledger payload"),
     ("ARTICLES_JSON", len(digests_json), None, f" · all {len(items)} articles, digests only"),
     ("FULL_TEXT", len(fulltext), None, f" · {len(fulltext_ids)} articles verbatim"),
     ("(every body in full)", all_bodies, None, " · what the digests replaced")],
    f"What the scoring model reads - one {ai_client.MODEL_NAME} call",
    "characters sent. The last bar is the counterfactual, not something sent.",
    fmt="{:,.0f}",
))

focus = max(digested, key=lambda it: it.get("stage1_severity") or 0.0)
print(f"ONE DIGEST IN FULL - {focus['id']}, the highest-severity article "
      f"({len(digest_engine.article_input_text(focus)):,} chars in, "
      f"{len(json.dumps(focus['digest'])):,} out)\n")
print(json.dumps(focus["digest"], indent=2, ensure_ascii=False))

ONE DIGEST IN FULL - a19, the highest-severity article (1,948 chars in, 468 out)

{
  "what_happened": "British cyclist Finlay Tarling died after succumbing to injuries sustained in a crash in the Volta a Portugal.",
  "actors": "Finlay Tarling collided with a vehicle not associated with the race that entered the course against traffic.",
  "numbers": "1 casualty, age 19, 20km remaining in the race, 3 family members mentioned (parents Michael and Dawn, brother Josh)",
  "transmission": "not stated",
  "directly_about_country": true,
  "stage1_severity": 85
}


## 3 · The score

One structured-output call returns four ledger scores, two horizon scores,
condition flags and a per-article impact. It holds the rubric the mini model never
saw — the ledger definitions, the bands, the calibration anchors, the three-door
event test — and it is the only model that outputs a risk score.

It never raises. Without `OPENAI_API_KEY`, or on a network or parse failure, it
returns `score=None` and the rest of the notebook still runs with empty tables.

In [9]:
llm_output = langchain_llm.country_llm_score(
    country_display=NAME,
    payload=evidence,          # the three-ledger payload, not the panel one
    articles=items,
    as_of=AS_OF,
    fulltext_ids=fulltext_ids,
)

display(scorecard(
    NAME, AS_OF, llm_output.get("score"), llm_output.get("score_3m"),
    non_investable=bool(llm_output.get("non_investable")),
    summary=llm_output.get("bullet_summary") or "",
))

INFO    httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [10]:
ledgers = llm_output.get("ledger_scores") or {}

display(hbar(
    [("friction", ledgers.get("friction"), "friction", " · higher = a worse wedge"),
     ("order-uncertainty", ledgers.get("order_uncertainty"), "uncertainty",
      " · higher = less underwritable"),
     ("information", ledgers.get("information_capacity"), "information",
      " · higher = weaker instruments"),
     ("edge vitality", ledgers.get("edge_vitality"), "edge",
      " · NOT risk - higher = more vitality")],
    f"The four ledgers - {NAME}",
    "0-1. The first three are risk and point the same way. The fourth is not, and "
    "may never raise a horizon score.",
    max_value=1.0,
    footer="Edge vitality is reported, never penalised: churn, startup formation AND "
           "failure, and human-capital formation are the system learning. A country "
           "where nothing is created and nothing fails is not stable, it is inert.",
))

### Why — the model's own citations

`subscore_evidence` is the model naming what moved each risk ledger, and
`news_article_scores` is what it made of each article. Articles covering the same
underlying event share a `topic_group`, and within a group only the highest impact
counts — so three write-ups of one protest cannot outweigh three separate crises.

In [11]:
evid = llm_output.get("subscore_evidence") or {}
display(notes(
    [("friction", "Friction", evid.get("friction") or []),
     ("uncertainty", "Order-uncertainty", evid.get("order_uncertainty") or []),
     ("information", "Information", evid.get("information_capacity") or [])],
    f"What the model cited - {NAME}",
    "its own words, stored in risk_snapshot.subscore_evidence. Edge vitality carries "
    "no citation list: it is a reading, not a charge sheet.",
))

imp_map, topic_map = article_ranking.impact_topic_maps(llm_output)
items_by_id = {it["id"]: it for it in items if isinstance(it, dict) and it.get("id")}
top_ids = article_ranking.select_top_ids(items_by_id, imp_map, topic_map, ISO2)

display(hbar(
    [(f"{'▸ ' if aid in top_ids else ''}{aid} · "
      f"{(topic_map.get(aid) or 'ungrouped')[:26]}",
      imp_map.get(aid), None,
      f" · {(items_by_id[aid].get('title') or '')[:90]}")
     for aid in sorted(items_by_id, key=lambda a: imp_map.get(a) or 0, reverse=True)],
    f"Per-article impact - {NAME}",
    "0-1, grouped by the topic the model assigned. ▸ marks the three that reach "
    "the dashboard. Hover for the headline.",
    max_value=1.0,
    footer=f"{len(set(topic_map.values()))} distinct topics across {len(items_by_id)} "
           f"articles. With 3 or more topics the best article of each of the top 3 wins, "
           f"so the dashboard shows three stories rather than three takes on one.",
))

INFO    backend.news_fetching.article_ranking: [PT] AI identified 15 topics (used 1/article).


## 4 · Guardrails — observe, don't enforce

This step used to be the enforcement layer: condition-flag floors, inflation
tiers, a political-stability cap, and a sanctions gate that forced a score to
`1.0`. All of it is deleted.

What survived is the distinction between a **floor** and a **badge**. A floor was
a claim about *risk*, made by overwriting the model's judgement with a number from
a YAML file — afterwards nobody could tell which half of a stored score came from
where. A badge is a claim about *law*: whether US persons may lawfully hold a
country's securities is a fact about the sanctions regime, not an opinion about
risk, and it belongs beside the score rather than inside it.

Contradictions still get noticed. When the model flags an active war and then
scores the country 44, `utils/lint.py` writes both down next to each other and
lets a human look. Its thresholds are advisory tripwires that encode no view about
what a score *should* be — nothing reads them back to change one.

In [12]:
audit = pd.DataFrame([
    {"field": "score_12m", "model_said": llm_output.get("raw_score_12m"),
     "stored": llm_output.get("score")},
    {"field": "score_3m", "model_said": llm_output.get("raw_score_3m"),
     "stored": llm_output.get("score_3m")},
]).set_index("field")
audit["moved_by"] = (pd.to_numeric(audit["stored"], errors="coerce")
                     - pd.to_numeric(audit["model_said"], errors="coerce")).round(6)
display(audit)
assert (audit["moved_by"].fillna(0) == 0).all(), "a score was edited - that should be impossible"

print(f"non_investable  {llm_output.get('non_investable')}   "
      f"legal_gate {llm_output.get('legal_gate') or '(unrestricted at this as_of)'}")
print(f"coverage        {llm_output.get('evidence_coverage')}  "
      f"(how completely the evidence captures the situation)")
print(f"flags           {llm_output.get('condition_flags')}")

findings = lint.check(
    country_iso2=ISO2, as_of=AS_OF,
    condition_flags=llm_output.get("condition_flags"),
    # lint's tripwires are on the model's 0-100 grid; stored values are 0-1.
    score_3m=round((llm_output.get("score_3m") or 0) * 100) or None,
    score_12m=round((llm_output.get("score") or 0) * 100) or None,
    ledger_scores={k: round(v * 100) for k, v in ledgers.items() if v is not None},
    suppressed_vol_flag=evidence["uncertainty_inputs"]["suppressed_vol_flag"]["value"],
    non_investable=bool(llm_output.get("non_investable")),
)
print(f"\ntripwires       war<{lint.WAR_SCORE_FLOOR}  "
      f"sovereign_stress<{lint.SOVEREIGN_STRESS_SCORE_FLOOR}  "
      f"suppressed_calm<{lint.SUPPRESSED_CALM_UNCERTAINTY_FLOOR}  (advisory)")
if findings:
    lint.log_findings(findings)
    display(pd.DataFrame(findings)[["rule", "detail"]])
else:
    print("lint            no findings - the model's flags and its scores agree.")

# What the model saw, hashed, so a stored score is reproducible: two hashes per
# article because "what we held" and "what the model read" are different
# questions. The pipeline writes this to risk_snapshot.input_manifest.
input_manifest = provenance.build_input_manifest(
    items=items, prompt_entries=langchain_llm.prompt_entries(items),
    fulltext_ids=fulltext_ids, payload=payload,
    model_id=llm_output.get("model_id"), prompt_version=llm_output.get("prompt_version"),
    policy_version=llm_output.get("policy_version"), seed=ai_client.SEED,
)
v = input_manifest["macro_vintages"]
print(f"provenance      {len(input_manifest['articles'])} articles hashed, panel "
      f"{v['vintage_scheme']} generated {v['panel_generated_at']}, newest year "
      f"{v['latest_year']}, git {input_manifest['git_sha'] or '(GIT_SHA unset)'}")

,model_said,stored,moved_by
field,,,
score_12m,0.40,0.40,0.0
score_3m,0.38,0.38,0.0


non_investable  False   legal_gate (unrestricted at this as_of)
coverage        0.75  (how completely the evidence captures the situation)
flags           {'war_on_territory': False, 'internal_conflict_level': 'none', 'emergency_rule': False, 'sovereign_stress': False}

tripwires       war<70  sovereign_stress<55  suppressed_calm<40  (advisory)
lint            no findings - the model's flags and its scores agree.
provenance      20 articles hashed, panel as-published-latest generated 2026-08-15T03:11Z, newest year 2025, git (GIT_SHA unset)


## 5 · What reaches the dashboard

The three winning articles, with one last attempt at a thumbnail through
Crawlbase (a credit per call, hence the Top-3-only scope). These are the rows
`risk_snapshot_article` would receive.

**The notebook stops before the write.** `upsert_snapshot` is never called, so a
run can never overwrite today's real snapshot. Use
`backend/tests/live_country_check.py` when you want the write plus verification
and cleanup.

In [13]:
article_enrichment.enrich_top_images(top_ids, items_by_id)
top_articles = article_ranking.build_top_articles(top_ids, items_by_id, imp_map)

display(HTML(_CSS + '<div class="viz"><h4>' + f"Top 3 - {NAME}" + "</h4>" + "".join(
    '<div style="display:flex;gap:12px;margin:12px 0;align-items:flex-start">'
    + (f'<img src="{a["image"]}" style="width:150px;border-radius:5px">' if a["image"] else "")
    + f'<div><b>#{a["rank"]} &middot; impact {a["impact"]}</b><br>'
      f'<a href="{a["url"]}" target="_blank">{_esc(a["title"])}</a><br>'
      f'<small style="color:var(--ink2)">{_esc(a["source"])} &middot; {a["published_at"]}</small>'
      f'<br><small style="color:var(--ink2)">{_esc((a["summary"] or "")[:240])}</small></div></div>'
    for a in top_articles) + "</div>"))

pd.DataFrame(top_articles).set_index("rank")[["impact", "source", "published_at", "title"]]

,impact,source,published_at,title
rank,,,,
1,0.85,Outside Magazine,2026-08-14T17:45:28Z,19-Year-Old Development Racer Finlay Tarling Dies in Crash at Volta a Portug...
2,0.60,Portugal Resident,2026-08-12T14:56:39Z,Luís Neves once again refers to “civil war” on Portuguese roads - Portugal R...
3,0.60,portugoal.net,2026-07-16T07:00:00Z,End of the line for Boavista as court ruling forces club to close down - por...


## Summary

That is the whole of rating one country: `pipeline._process_country` from the
macro panel to the upsert, in order.

What it leaves out are the other phases of `main.py`, none of which touch a
country's score — the economic-calendar and market-price fetches, the IMF
recent-indicator refresh, and the post-loop global alert ranking that pools every
country's Top-3 and re-ranks them for the dashboard's alert strip.

In [14]:
print(f"{NAME} ({ISO2})   risk {llm_output.get('score')} (12m), "
      f"{llm_output.get('score_3m')} (3m)   as of {AS_OF}")
print(f"  ledgers        " + "  ".join(f"{k}={v}" for k, v in ledgers.items()))
print(f"  evidence       {present} indicators, {len(evidence['computed'])} computed "
      f"metrics, {late} of {len(behind)} at 2.0× stale or worse")
print(f"  articles       {fetched_n} fetched, {len(digested)} digested, "
      f"{len(fulltext_ids)} read in full, {len(top_articles)} shown")
print(f"  guardrails     non_investable={llm_output.get('non_investable')}, "
      f"{len(findings)} lint finding(s), 0 scores edited")
print(f"  stamps         model {llm_output.get('model_id')}   "
      f"prompt {llm_output.get('prompt_version')}   "
      f"policy {llm_output.get('policy_version')}")

Portugal (PT)   risk 0.4 (12m), 0.38 (3m)   as of 2026-08-15
  ledgers        friction=0.35  order_uncertainty=0.3  information_capacity=0.15  edge_vitality=0.65
  evidence       23 indicators, 7 computed metrics, 4 of 22 at 2.0× stale or worse
  articles       20 fetched, 20 digested, 3 read in full, 3 shown
  guardrails     non_investable=False, 0 lint finding(s), 0 scores edited
  stamps         model gpt-4o-2024-08-06   prompt v4.0-masked-production   policy p2.0-observe-only
